In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib

from IPython.display import display

# Project root (assuming notebook is inside notebooks/)
ROOT = Path.cwd().parent

# Data directories
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

# Output directories
TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"

# Create output folders if they do not exist
for folder in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Dataset paths
HOUSEHOLD_FILE = RAW_DIR / "ghs-2025-household-v1.csv"
PERSON_FILE = RAW_DIR / "ghs-2025-person-v1.csv"

# Verify files exist
assert HOUSEHOLD_FILE.exists(), f"Missing file: {HOUSEHOLD_FILE}"
assert PERSON_FILE.exists(), f"Missing file: {PERSON_FILE}"

print("Project directories configured successfully.")
print(f"Project root: {ROOT}")
print(f"Household dataset: {HOUSEHOLD_FILE.name}")
print(f"Person dataset: {PERSON_FILE.name}")

Project directories configured successfully.
Project root: c:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality
Household dataset: ghs-2025-household-v1.csv
Person dataset: ghs-2025-person-v1.csv


In [2]:
# Read datasets
household_raw = pd.read_csv(
    HOUSEHOLD_FILE,
    encoding="cp1252",
    low_memory=False,
    dtype=str
)

person_raw = pd.read_csv(
    PERSON_FILE,
    encoding="cp1252",
    low_memory=False,
    dtype=str
)

print("Datasets loaded successfully.\n")

print("Household dataset:", household_raw.shape)
print("Person dataset:", person_raw.shape)

print("\nHousehold preview:")
display(household_raw.head())

print("\nPerson preview:")
display(person_raw.head())

Datasets loaded successfully.

Household dataset: (20095, 207)
Person dataset: (67748, 124)

Household preview:


,uqnr,personnr,psu,prov,head_popgrp,head_sex,head_age,hsg_maind,hsg_walls,hsg_roof,...,sevdisab_hh,econact_hh,totmhinc,stratum,geotype,metro,metro_code,rotation,house_wgt,Metro_code
0,160100090000001901,01,16010009000,1. Western Cape,4. White,2. Female,67. 67,1. Dwelling/house or brick/concrete block stru...,1. Bricks,2. Corrugated iron/zinc,...,0. 0,0. 0,1990. 1990,10301,3,2. Non-metro,01,3. Quarter 3,879.092961117,WC â€“ Non-metro
1,160100090000008501,01,16010009000,1. Western Cape,2. Coloured,1. Male,64. 64,6. Semi-detached house,2. Cement block/concrete,2. Corrugated iron/zinc,...,0. 0,0. 0,1990. 1990,10301,3,2. Non-metro,01,3. Quarter 3,874.343152544,WC â€“ Non-metro
2,160100090000011801,01,16010009000,1. Western Cape,4. White,1. Male,42. 42,1. Dwelling/house or brick/concrete block stru...,1. Bricks,2. Corrugated iron/zinc,...,1. 1,1. 1,35417. 35417,10301,3,2. Non-metro,01,3. Quarter 3,1370.42498741,WC â€“ Non-metro
3,160100090000015101,01,16010009000,1. Western Cape,4. White,1. Male,27. 27,1. Dwelling/house or brick/concrete block stru...,1. Bricks,11. Asbestos,...,0. 0,2. 2,55010. 55010,10301,3,2. Non-metro,01,3. Quarter 3,1629.96290366,WC â€“ Non-metro
4,160100250000001501,01,16010025000,1. Western Cape,2. Coloured,2. Female,38. 38,1. Dwelling/house or brick/concrete block stru...,1. Bricks,8. Tile,...,0. 0,0. 0,20000. 20000,10111,1,2. Non-metro,01,4. Quarter 4,1410.12165539,WC â€“ Non-metro



Person preview:


,uqnr,personnr,psu,prov,Four_by_Four,Sex,age,age_grp1,Population,Languages,...,metro,stratum,lab_salary,employ_Status1,employ_Status2,rotation,person_wgt,UqNr,Age_grp1,Metro_code
0,160100090000001901,01,16010009000,1. Western Cape,1. Yes,2. Female,67. 67,14,4. White,1. Afrikaans,...,2. Non-metro,10301,888888888. Not applicable,9. Unspecified,3. Not economically active,3. Quarter 3,782.0718251660001,1. 160100090000001901,6. 25â€“29 years,WC â€“ Non-metro
1,160100090000008501,01,16010009000,1. Western Cape,1. Yes,1. Male,64. 64,13,2. Coloured,1. Afrikaans,...,2. Non-metro,10301,888888888. Not applicable,9. Unspecified,3. Not economically active,3. Quarter 3,725.027502607,2. 160100090000008501,5. 20â€“24 years,WC â€“ Non-metro
2,160100090000011801,01,16010009000,1. Western Cape,1. Yes,1. Male,42. 42,9,4. White,1. Afrikaans,...,2. Non-metro,10301,33427. 33427,1. Employed,1. Employed,3. Quarter 3,851.73692763,3. 160100090000011801,16. 75+,WC â€“ Non-metro
3,160100090000011801,02,16010009000,1. Western Cape,1. Yes,1. Male,66. 66,14,4. White,1. Afrikaans,...,2. Non-metro,10301,888888888. Not applicable,9. Unspecified,3. Not economically active,3. Quarter 3,851.73692763,3. 160100090000011801,6. 25â€“29 years,WC â€“ Non-metro
4,160100090000015101,01,16010009000,1. Western Cape,1. Yes,1. Male,27. 27,6,4. White,1. Afrikaans,...,2. Non-metro,10301,33427. 33427,1. Employed,1. Employed,3. Quarter 3,979.758375057,4. 160100090000015101,13. 60â€“64 years,WC â€“ Non-metro


In [3]:
# Calculate SHA-256 file hash
def file_hash(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


# Create source inventory
inventory = []

for name, df, path in [
    ("Household", household_raw, HOUSEHOLD_FILE),
    ("Person", person_raw, PERSON_FILE)
]:
    inventory.append({
        "Dataset": name,
        "Filename": path.name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate_Rows": df.duplicated().sum(),
        "Missing_uqnr": df["uqnr"].isna().sum(),
        "Unique_Households": df["uqnr"].nunique(),
        "SHA256": file_hash(path)
    })

# Convert to DataFrame
T01 = pd.DataFrame(inventory)

# Display table
display(T01)

# Export table
T01.to_csv(
    TABLES_DIR / "T01_source_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

print("T01_source_inventory.csv exported successfully.")

,Dataset,Filename,Rows,Columns,Duplicate_Rows,Missing_uqnr,Unique_Households,SHA256
0,Household,ghs-2025-household-v1.csv,20095,207,0,0,20095,fe2adaf1ce8a864ab41887de180969441a30dd7711e8ee...
1,Person,ghs-2025-person-v1.csv,67748,124,0,0,20095,e58eacdc18efcc3905097414d509f1a2fb5b1050216622...


T01_source_inventory.csv exported successfully.


In [4]:
# 1. Check identifier uniqueness
duplicate_households = household_raw["uqnr"].duplicated().sum()

duplicate_person_ids = person_raw.duplicated(
    subset=["uqnr", "personnr"]
).sum()

missing_personnr = person_raw["personnr"].isna().sum()


# 2. Check household-person linkage
household_ids = set(household_raw["uqnr"])
person_household_ids = set(person_raw["uqnr"])

households_without_persons = len(
    household_ids - person_household_ids
)

persons_without_households = (
    ~person_raw["uqnr"].isin(household_ids)
).sum()


# 3. Count linked persons per household
person_counts = (
    person_raw.groupby("uqnr")
    .size()
    .rename("linked_persons")
)

household_check = household_raw[
    ["uqnr", "hholdsz"]
].copy()

household_check = household_check.merge(
    person_counts,
    on="uqnr",
    how="left",
    validate="one_to_one"
)


# 4. Extract numeric household size
household_check["reported_size"] = pd.to_numeric(
    household_check["hholdsz"].str.extract(
        r"^\s*(\d+)\s*\."
    )[0],
    errors="coerce"
)

household_check["size_match"] = (
    household_check["reported_size"]
    == household_check["linked_persons"]
)

valid_size = household_check["reported_size"].notna()

size_mismatches = (
    valid_size & ~household_check["size_match"]
).sum()

missing_household_size = (~valid_size).sum()


# 5. Create linkage integrity table
T02 = pd.DataFrame({
    "Check": [
        "Duplicate household identifiers",
        "Duplicate person identifiers",
        "Missing person identifiers",
        "Households without linked persons",
        "Person records without households",
        "Missing or invalid household size",
        "Household size mismatches"
    ],
    "Count": [
        duplicate_households,
        duplicate_person_ids,
        missing_personnr,
        households_without_persons,
        persons_without_households,
        missing_household_size,
        size_mismatches
    ]
})

display(T02)


# 6. Export results
T02.to_csv(
    TABLES_DIR / "T02_linkage_integrity.csv",
    index=False,
    encoding="utf-8-sig"
)

print("T02_linkage_integrity.csv exported successfully.")

,Check,Count
0,Duplicate household identifiers,0
1,Duplicate person identifiers,0
2,Missing person identifiers,0
3,Households without linked persons,0
4,Person records without households,0
5,Missing or invalid household size,0
6,Household size mismatches,0


T02_linkage_integrity.csv exported successfully.


In [5]:
# Variables required for our study
household_vars = [
    "uqnr", "psu", "stratum", "house_wgt",
    "prov", "geotype", "metro",
    "hhw_time", "hhw_hltfac", "hhw_transp",
    "hwl_vehicle", "totmhinc", "fin_exp",
    "fsd_worried", "fsd_healthy", "fsd_fewfoods",
    "fsd_skipped", "fsd_ateless", "fsd_ranout",
    "fsd_hungry", "fsd_whlday"
]

person_vars = [
    "uqnr", "personnr", "psu", "stratum",
    "person_wgt", "hlt_medi",
    "age", "Sex", "education", "employ_Status1"
]


# Function to audit variables
def audit_variables(df, variables, dataset_name):

    results = []

    for var in variables:

        if var not in df.columns:
            results.append({
                "Dataset": dataset_name,
                "Variable": var,
                "Data_Type": "NOT FOUND",
                "Missing_N": None,
                "Missing_Percent": None,
                "Unique_Values": None,
                "Observed_Values": "VARIABLE NOT FOUND"
            })
            continue

        column = df[var]

        # Most common observed values
        observed = (
            column.value_counts(dropna=False)
            .head(10)
            .index.astype(str)
            .tolist()
        )

        results.append({
            "Dataset": dataset_name,
            "Variable": var,
            "Data_Type": str(column.dtype),
            "Missing_N": int(column.isna().sum()),
            "Missing_Percent": round(
                column.isna().mean() * 100, 2
            ),
            "Unique_Values": column.nunique(dropna=True),
            "Observed_Values": " | ".join(observed)
        })

    return pd.DataFrame(results)


# Audit household and person variables
household_audit = audit_variables(
    household_raw, household_vars, "Household"
)

person_audit = audit_variables(
    person_raw, person_vars, "Person"
)

# Combine results
T03 = pd.concat(
    [household_audit, person_audit],
    ignore_index=True
)

# Display full table
display(T03)

# Export CSV
T03.to_csv(
    TABLES_DIR / "T03_missingness_and_codes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("T03_missingness_and_codes.csv exported successfully.")

,Dataset,Variable,Data_Type,Missing_N,Missing_Percent,Unique_Values,Observed_Values
0,Household,uqnr,str,0,0.0,20095,160100090000001901 | 160100090000008501 | 1601...
1,Household,psu,str,0,0.0,3226,79714653000 | 52610158000 | 19913145000 | 2871...
2,Household,stratum,str,0,0.0,248,77102 | 77116 | 30101 | 76112 | 76111 | 78113 ...
3,Household,house_wgt,str,0,0.0,4194,457.481875075 | 504.710895628 | 588.086029046 ...
4,Household,prov,str,0,0.0,9,7. Gauteng | 5. KwaZulu-Natal | 2. Eastern Cap...
5,Household,geotype,str,0,0.0,3,1 | 2 | 3
6,Household,metro,str,0,0.0,2,2. Non-metro | 1. Metro
7,Household,hhw_time,str,0,0.0,6,15â€“29 minutes | Less than 15 minutes | 30â€“...
8,Household,hhw_hltfac,str,0,0.0,14,2. Public sector: Clinic | 6. Private sector: ...
9,Household,hhw_transp,str,0,0.0,8,1. Walking | 2. Minibus taxi/sedan taxi/bakkie...


T03_missingness_and_codes.csv exported successfully.


In [6]:
# Create copies of survey design variables
hh_design = household_raw[
    ["uqnr", "psu", "stratum", "house_wgt"]
].copy()

person_design = person_raw[
    ["uqnr", "psu", "stratum", "person_wgt"]
].copy()


# Convert survey weights to numeric
hh_design["weight"] = pd.to_numeric(
    hh_design["house_wgt"], errors="coerce"
)

person_design["weight"] = pd.to_numeric(
    person_design["person_wgt"], errors="coerce"
)


# Function to validate survey design
def check_survey_design(df, dataset_name):

    # Number of distinct PSUs within each stratum
    psus_per_stratum = (
        df.groupby("stratum")["psu"].nunique()
    )

    # Check whether a PSU appears in multiple strata
    strata_per_psu = (
        df.groupby("psu")["stratum"].nunique()
    )

    return {
        "Dataset": dataset_name,
        "Records": len(df),
        "Strata": df["stratum"].nunique(),
        "PSUs": df["psu"].nunique(),
        "Missing_PSU": df["psu"].isna().sum(),
        "Missing_Stratum": df["stratum"].isna().sum(),
        "Invalid_Weights": (
            df["weight"].isna() | (df["weight"] <= 0)
        ).sum(),
        "Minimum_Weight": df["weight"].min(),
        "Maximum_Weight": df["weight"].max(),
        "Weight_Sum": df["weight"].sum(),
        "Singleton_Strata": (psus_per_stratum == 1).sum(),
        "PSUs_In_Multiple_Strata": (strata_per_psu > 1).sum()
    }


# Validate both datasets
survey_results = pd.DataFrame([
    check_survey_design(hh_design, "Household"),
    check_survey_design(person_design, "Person")
])

display(survey_results)


# Save audit results
survey_results.to_csv(
    TABLES_DIR / "T06_survey_design_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

print("T06_survey_design_audit.csv exported successfully.")

,Dataset,Records,Strata,PSUs,Missing_PSU,Missing_Stratum,Invalid_Weights,Minimum_Weight,Maximum_Weight,Weight_Sum,Singleton_Strata,PSUs_In_Multiple_Strata
0,Household,20095,248,3226,0,0,0,267.60605,5738.467872,2.011420e+07,0,0
1,Person,67748,248,3226,0,0,0,50.00000,6194.953311,6.407169e+07,0,0


T06_survey_design_audit.csv exported successfully.


In [7]:
# Examine all healthcare travel-time categories
travel_time = household_raw[
    ["hhw_time", "house_wgt"]
].copy()

# Convert weights to numeric
travel_time["weight"] = pd.to_numeric(
    travel_time["house_wgt"],
    errors="coerce"
)

# Unweighted frequency
counts = travel_time["hhw_time"].value_counts(
    dropna=False
)

# Weighted frequency
weighted_counts = travel_time.groupby(
    "hhw_time", dropna=False
)["weight"].sum()

# Create summary table
travel_summary = pd.DataFrame({
    "Unweighted_N": counts,
    "Weighted_N": weighted_counts
}).fillna(0)

# Calculate percentages
travel_summary["Unweighted_Percent"] = (
    travel_summary["Unweighted_N"]
    / len(travel_time) * 100
)

travel_summary["Weighted_Percent"] = (
    travel_summary["Weighted_N"]
    / travel_time["weight"].sum() * 100
)

# Display results
display(travel_summary.round(2))

,Unweighted_N,Weighted_N,Unweighted_Percent,Weighted_Percent
hhw_time,,,,
15â€“29 minutes,9180,9004383.68,45.68,44.77
30â€“89 minutes,2948,2691280.05,14.67,13.38
90 minutes and more,236,217431.10,1.17,1.08
DO NOT KNOW,22,27133.30,0.11,0.13
Less than 15 minutes,7707,8171318.73,38.35,40.62
Unspecified,2,2649.35,0.01,0.01


In [8]:
# Create working household dataset
household = household_raw.copy()

# Correct encoding artefact in travel-time labels
household["hhw_time_clean"] = (
    household["hhw_time"]
    .str.replace("â€“", "–", regex=False)
    .str.strip()
)

# Define travel-time categories
travel_categories = {
    "Less than 15 minutes": "Under 15 minutes",
    "15–29 minutes": "15–29 minutes",
    "30–89 minutes": "30–89 minutes",
    "90 minutes and more": "90 minutes or more"
}

household["travel_time_group"] = (
    household["hhw_time_clean"].map(travel_categories)
)

# Create binary outcome: 30 minutes or more
household["travel30_hh"] = np.where(
    household["travel_time_group"].isin(
        ["30–89 minutes", "90 minutes or more"]
    ),
    1,
    np.where(
        household["travel_time_group"].isin(
            ["Under 15 minutes", "15–29 minutes"]
        ),
        0,
        np.nan
    )
)

# Convert household weight to numeric
household["house_wgt_num"] = pd.to_numeric(
    household["house_wgt"],
    errors="coerce"
)

# Check outcome distribution
outcome_summary = (
    household.groupby("travel30_hh", dropna=False)
    .agg(
        Households=("uqnr", "count"),
        Weighted_Households=("house_wgt_num", "sum")
    )
    .reset_index()
)

display(outcome_summary)

# Verify outcome coding
assert household["travel30_hh"].notna().sum() == 20071
assert household["travel30_hh"].isna().sum() == 24

print("Primary outcome created and validated successfully.")

,travel30_hh,Households,Weighted_Households
0,0.0,16887,1.717570e+07
1,1.0,3184,2.908711e+06
2,NaN,24,2.978264e+04


Primary outcome created and validated successfully.
